# ※ 과제 안내
- 과제 배점: 각 문제당 10점, 총점 100점입니다. 부분 점수는 제공되지 않습니다.

- 채점 기준:
    - 출력 결과 일치: 제출한 코드가 제시된 출력 결과와 일치하는 경우에만 정답으로 인정됩니다.
    - 코드의 다양성 인정: 출력 결과가 동일하다면 다양한 접근 방식을 존중하여 정답으로 인정합니다.

---
# MCP 실전 과제 — Part 1: 코드 구현 (70점)

Chapter 2~4 핵심 구현 능력을 평가합니다.
각 문제 10점, 총 7문제 70점입니다.

In [ ]:
# 실습 환경 세팅

!pip install -q pyyaml

In [ ]:
# 공통 실행 환경 (모든 문제 공통)

import json
import sqlite3
import hashlib
import re
import time
import asyncio
import inspect
import csv
import io
import os
import tempfile
import typing
import yaml
from pathlib import Path
from dataclasses import dataclass, field, asdict
from enum import Enum
from datetime import datetime, timezone

---
## 문제 1. Tool 기초: 타입 힌트와 JSON Schema [10점]

In [ ]:
# 문제 1-1. 타입 힌트 → JSON Schema 변환 함수

def python_type_to_json_schema(py_type) -> str:
    mapping = {
        str: "string",
        int: "integer",
        float: "number",
        bool: "boolean",
    }
    return mapping.get(py_type, "string")

In [ ]:
# 문제 1-2. 함수 시그니처에서 MCP Tool 스키마 추출

def extract_tool_schema(func) -> dict:
    sig = inspect.signature(func)
    properties = {}
    required = []

    for name, param in sig.parameters.items():
        if name == "ctx":
            continue

        annotation = param.annotation
        if annotation == inspect.Parameter.empty:
            json_type = "string"
        else:
            json_type = python_type_to_json_schema(annotation)

        properties[name] = {"type": json_type}

        if param.default == inspect.Parameter.empty:
            required.append(name)

    return {
        "name": func.__name__,
        "description": (func.__doc__ or "").strip(),
        "inputSchema": {
            "type": "object",
            "properties": properties,
            "required": required,
        },
    }

In [ ]:
# 검증 1

def lookup_inventory(query: str, limit: int = 10, ctx=None) -> str:
    """Search inventory items by keyword."""
    pass

schema = extract_tool_schema(lookup_inventory)
print(json.dumps(schema, indent=2))

---
## 문제 2. 실전 Tool (1): CSV → SQLite 재고 조회 [10점]

In [ ]:
# 제공 데이터 (수정 금지)

INVENTORY_CSV = """item_id,name,category,quantity,location,status,last_updated
INV-001,MacBook Pro 14 M3 Pro,Electronics,25,Warehouse A,in_stock,2026-01-15
INV-002,Dell Latitude 5540,Electronics,30,Warehouse B,in_stock,2026-01-20
INV-003,Lenovo ThinkPad X1 Carbon,Electronics,15,Warehouse A,in_stock,2026-01-18
INV-004,Dell UltraSharp U2723QE Monitor,Electronics,40,Warehouse C,in_stock,2026-01-10
INV-005,Logitech MX Keys Keyboard,Peripherals,100,Warehouse B,in_stock,2026-01-22
INV-006,Apple Magic Mouse,Peripherals,60,Warehouse A,in_stock,2026-01-12
INV-007,Jabra Evolve2 75 Headset,Peripherals,45,Warehouse C,in_stock,2026-01-25
INV-008,CalDigit TS4 Docking Station,Peripherals,20,Warehouse A,low_stock,2026-01-08
INV-009,iPad Air M2,Electronics,10,Warehouse B,low_stock,2026-01-14
INV-010,HP LaserJet Pro M404dn,Office,8,Warehouse C,in_stock,2026-01-30
INV-011,Office Chair ErgoMax Pro,Furniture,50,Warehouse A,in_stock,2026-02-01
INV-012,Standing Desk VariDesk Pro,Furniture,12,Warehouse B,low_stock,2026-01-28"""

In [ ]:
# 문제 2-1. CSV → SQLite 인메모리 DB 초기화

def init_inventory_db(csv_data: str) -> sqlite3.Connection:
    db = sqlite3.connect(":memory:")
    db.row_factory = sqlite3.Row

    db.execute("""
        CREATE TABLE inventory (
            item_id TEXT PRIMARY KEY,
            name TEXT NOT NULL,
            category TEXT NOT NULL,
            quantity INTEGER NOT NULL,
            location TEXT NOT NULL,
            status TEXT NOT NULL,
            last_updated TEXT NOT NULL
        )
    """)

    reader = csv.DictReader(io.StringIO(csv_data.strip()))
    for row in reader:
        db.execute(
            "INSERT INTO inventory VALUES (?, ?, ?, ?, ?, ?, ?)",
            (row["item_id"], row["name"], row["category"],
             int(row["quantity"]), row["location"], row["status"],
             row["last_updated"])
        )
    db.commit()
    return db

In [ ]:
# 문제 2-2. SQL LIKE 검색 함수

def search_inventory(db: sqlite3.Connection, query: str) -> list[dict]:
    pattern = f"%{query.strip()}%"
    rows = db.execute(
        """
        SELECT item_id, name, category, quantity, location, status, last_updated
        FROM inventory
        WHERE LOWER(name) LIKE LOWER(?) OR LOWER(category) LIKE LOWER(?)
        LIMIT 10
        """,
        (pattern, pattern)
    ).fetchall()
    return [dict(row) for row in rows]

In [ ]:
# 검증 2

db = init_inventory_db(INVENTORY_CSV)

results = search_inventory(db, "dell")
print(f"'dell' 검색 결과: {len(results)}건")
for r in results:
    print(f"  - {r['item_id']}: {r['name']} ({r['category']})")

results2 = search_inventory(db, "peripherals")
print(f"\n'peripherals' 검색 결과: {len(results2)}건")

results3 = search_inventory(db, "xyz_not_exist")
print(f"\n'xyz_not_exist' 검색 결과: {len(results3)}건")

---
## 문제 3. 에러 처리: ToolError 패턴 [10점]

In [ ]:
# 문제 3-1. ErrorCode Enum과 ToolError 클래스

class ErrorCode(Enum):
    NOT_FOUND = "NOT_FOUND"
    INVALID_ARGUMENT = "INVALID_ARGUMENT"
    PERMISSION_DENIED = "PERMISSION_DENIED"
    INTERNAL_ERROR = "INTERNAL_ERROR"
    CONFLICT = "CONFLICT"

class ToolError(Exception):
    def __init__(self, code: ErrorCode, message: str):
        self.code = code
        self.message = message
        super().__init__(str(self))

    def __str__(self):
        return f"[{self.code.value}] {self.message}"

In [ ]:
# 문제 3-2. 안전한 검색 함수

def safe_search(db: sqlite3.Connection, query: str) -> list[dict]:
    if not query or not query.strip():
        raise ToolError(ErrorCode.INVALID_ARGUMENT, "검색어를 입력하세요")

    try:
        return search_inventory(db, query)
    except sqlite3.Error as e:
        raise ToolError(ErrorCode.INTERNAL_ERROR, f"데이터베이스 오류: {e}")

In [ ]:
# 검증 3

results = safe_search(db, "macbook")
print(f"정상 검색: {len(results)}건 - {results[0]['name']}")

empty = safe_search(db, "xyz")
print(f"빈 결과: {empty} (type: {type(empty).__name__})")

try:
    safe_search(db, "")
except ToolError as e:
    print(f"빈 입력 에러: {e}")

try:
    safe_search(db, "   ")
except ToolError as e:
    print(f"공백 입력 에러: {e}")

---
## 문제 4. 실전 Tool (2): YAML 파싱과 관련도 랭킹 [10점]

In [ ]:
# 제공 데이터 (수정 금지)

POLICY_DOC_1 = """---
title: VPN 설정 가이드
tags: [vpn, 네트워크, 보안, 원격근무]
last_updated: 2026-01-15
---
# VPN 설정 가이드

## 개요
사내 VPN은 WireGuard를 기본으로 사용합니다.

## 설치 방법
macOS에서는 Homebrew로 설치합니다.
VPN 연결이 필요한 경우 IT팀에 문의하세요.
VPN 설정 후 반드시 연결 테스트를 수행하세요."""

POLICY_DOC_2 = """---
title: 보안 정책
tags: [보안, 비밀번호, 인증, 컴플라이언스]
last_updated: 2026-02-01
---
# 보안 정책

## 비밀번호 규칙
비밀번호는 14자 이상이어야 합니다.

## MFA 필수
모든 직원은 MFA를 활성화해야 합니다.
VPN 접속 시에도 MFA 인증이 필요합니다."""

POLICY_DOC_3 = """---
title: 원격근무 정책
tags: [원격근무, 재택, 하이브리드]
last_updated: 2026-01-20
---
# 원격근무 정책

## 핵심 근무 시간
10:00~16:00 사이에는 반드시 연락 가능해야 합니다.

## 장비 지원
노트북, 모니터 1대, 키보드, 마우스, 헤드셋을 지원합니다."""

In [ ]:
# 문제 4-1. YAML Front-matter 파싱

def parse_frontmatter(content: str) -> tuple[dict, str]:
    pattern = re.compile(r"^---\s*\n(.*?)\n---\s*\n(.*)$", re.DOTALL)
    match = pattern.match(content)
    if not match:
        return {}, content
    meta = yaml.safe_load(match.group(1))
    body = match.group(2)
    return meta, body

In [ ]:
# 문제 4-2. 관련도 점수 계산

def calculate_relevance(query: str, title: str, tags: list[str], body: str) -> float:
    q = query.lower()
    score = 0.0

    if q in title.lower():
        score += 3.0

    if any(q in tag.lower() for tag in tags):
        score += 2.0

    body_count = body.lower().count(q)
    score += min(body_count * 0.5, 3.0)

    return score

In [ ]:
# 검증 4

meta1, body1 = parse_frontmatter(POLICY_DOC_1)
print(f"제목: {meta1['title']}")
print(f"태그: {meta1['tags']}")
print(f"본문 길이: {len(body1)}자")

docs = [POLICY_DOC_1, POLICY_DOC_2, POLICY_DOC_3]
print("\n=== 'vpn' 검색 관련도 ===")
for i, doc in enumerate(docs, 1):
    meta, body = parse_frontmatter(doc)
    score = calculate_relevance("vpn", meta["title"], meta["tags"], body)
    print(f"문서{i} ({meta['title']}): {score}점")

---
## 문제 5. 실전 Tool (3): Confirm Gate와 멱등성 [10점]

In [ ]:
# 문제 5-1. Confirm Gate 패턴 구현

def create_ticket(
    title: str,
    body: str,
    priority: str,
    confirm: bool,
    tickets_store: list
) -> str | dict:
    if not confirm:
        return f"[미리보기] 제목: {title} | 우선순위: {priority} — confirm=True로 생성하세요."

    ticket_id = f"TKT-{len(tickets_store) + 1:03d}"
    ticket = {
        "ticket_id": ticket_id,
        "title": title,
        "body": body,
        "priority": priority,
        "status": "open",
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
    tickets_store.append(ticket)
    return ticket

In [ ]:
# 문제 5-2. 멱등성 키 로직 추가

def create_ticket_idempotent(
    title: str,
    body: str,
    priority: str,
    confirm: bool,
    tickets_store: list
) -> str | dict:
    if not confirm:
        return f"[미리보기] 제목: {title} | 우선순위: {priority} — confirm=True로 생성하세요."

    idempotency_key = hashlib.sha256((title + body).encode()).hexdigest()[:16]

    for ticket in tickets_store:
        if ticket.get("idempotency_key") == idempotency_key:
            return ticket

    ticket_id = f"TKT-{len(tickets_store) + 1:03d}"
    ticket = {
        "ticket_id": ticket_id,
        "title": title,
        "body": body,
        "priority": priority,
        "status": "open",
        "created_at": datetime.now(timezone.utc).isoformat(),
        "idempotency_key": idempotency_key,
    }
    tickets_store.append(ticket)
    return ticket

In [ ]:
# 검증 5

store = []

preview = create_ticket_idempotent("프린터 고장", "2층 프린터가 작동하지 않습니다.", "medium", confirm=False, tickets_store=store)
print(f"미리보기: {preview}")
print(f"저장소 변경 없음: {len(store)}건")

ticket1 = create_ticket_idempotent("프린터 고장", "2층 프린터가 작동하지 않습니다.", "medium", confirm=True, tickets_store=store)
print(f"\n생성된 티켓: {ticket1['ticket_id']} - {ticket1['title']}")
print(f"저장소: {len(store)}건")

ticket2 = create_ticket_idempotent("프린터 고장", "2층 프린터가 작동하지 않습니다.", "medium", confirm=True, tickets_store=store)
print(f"\n중복 시도 결과: {ticket2['ticket_id']} (기존 티켓 반환)")
print(f"저장소 변경 없음: {len(store)}건")

ticket3 = create_ticket_idempotent("모니터 깜빡임", "3층 모니터가 깜빡입니다.", "high", confirm=True, tickets_store=store)
print(f"\n새 티켓: {ticket3['ticket_id']} - {ticket3['title']}")
print(f"저장소: {len(store)}건")

---
## 문제 6. 입력 검증과 보안 [10점]

LLM이 생성하는 입력은 신뢰할 수 없습니다.
Prompt Injection을 통해 악의적인 입력이 Tool에 전달될 수 있으므로,
모든 입력을 검증해야 합니다.

In [ ]:
# 제공 상수 (수정 금지)

CONTROL_CHARS = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")

DANGEROUS_PATTERNS = [
    re.compile(r"['\\";]\s*(?:DROP|DELETE|UPDATE|INSERT|ALTER|EXEC)", re.IGNORECASE),
    re.compile(r"<\s*script", re.IGNORECASE),
    re.compile(r"\$(?:gt|lt|ne|eq|regex|where|or|and)\b"),
    re.compile(r"__proto__|constructor\s*\["),
]

In [ ]:
# 문제 6-1. sanitize_string() 구현

def sanitize_string(value: str, max_length: int = 500) -> str:
    if not isinstance(value, str):
        raise ValueError("Input must be a string")
    value = value.strip()
    value = CONTROL_CHARS.sub("", value)
    value = value[:max_length]
    for pattern in DANGEROUS_PATTERNS:
        if pattern.search(value):
            raise ValueError("Dangerous pattern detected")
    return value

In [ ]:
# 문제 6-2. validate_doc_id() 구현

def validate_doc_id(doc_id: str) -> str:
    if not isinstance(doc_id, str):
        raise ValueError("doc_id must be a string")
    doc_id = doc_id.strip()
    if not doc_id:
        raise ValueError("doc_id cannot be empty")
    if "\x00" in doc_id:
        raise ValueError("doc_id contains null byte")
    pattern = re.compile(r"^[a-z0-9]([a-z0-9\-]{0,48}[a-z0-9])?$")
    if not pattern.match(doc_id):
        raise ValueError(f"Invalid doc_id: {doc_id}")
    return doc_id

In [ ]:
# 검증 6

print("=== sanitize_string ===")
print(f"정상: '{sanitize_string('  hello world  ')}'")
print(f"길이제한: '{sanitize_string('a' * 600, max_length=10)}'")

for attack in ["'; DROP TABLE users; --", "<script>alert(1)</script>", "$gt value"]:
    try:
        sanitize_string(attack)
        print(f"통과 (오류!): {attack}")
    except ValueError:
        print(f"차단됨: {attack[:30]}")

print("\n=== validate_doc_id ===")
for valid_id in ["vpn-setup", "remote-work", "a", "abc123"]:
    print(f"통과: {validate_doc_id(valid_id)}")

for bad_id in ["../../../etc/passwd", "foo/bar", "UPPER", "", "a" * 60, "hello\x00world"]:
    try:
        validate_doc_id(bad_id)
        print(f"통과 (오류!): {bad_id}")
    except ValueError:
        print(f"차단됨: {repr(bad_id)[:30]}")

# 기대 출력:
# === sanitize_string ===
# 정상: 'hello world'
# 길이제한: 'aaaaaaaaaa'
# 차단됨: '; DROP TABLE users; --
# 차단됨: <script>alert(1)</script>
# 차단됨: $gt value
# === validate_doc_id ===
# 통과: vpn-setup / remote-work / a / abc123
# 차단됨: (각 공격 패턴)

---
## 문제 7. Resource 구현 + Prompt 템플릿 [10점]

MCP Resource는 읽기 전용 데이터를 URI로 노출하고,
Prompt는 재사용 가능한 지시문 템플릿입니다.
정적 리소스와 동적 리소스를 구현하고, 프롬프트 템플릿을 작성합니다.

In [ ]:
# 제공 데이터 구조 (수정 금지)

@dataclass
class PolicyDoc:
    doc_id: str
    title: str
    path: Path
    tags: list[str]

In [ ]:
# 문제 7-1. 정책 인덱스 + 상세 리소스

def build_policy_index(policies: list[PolicyDoc]) -> list[dict]:
    return [{"doc_id": p.doc_id, "title": p.title, "tags": p.tags} for p in policies]

def get_policy_content(policies: list[PolicyDoc], doc_id: str, policy_dir: Path) -> dict:
    doc_id = validate_doc_id(doc_id)
    policy = None
    for p in policies:
        if p.doc_id == doc_id:
            policy = p
            break
    if policy is None:
        raise ValueError(f"Policy not found: {doc_id}")
    if not policy.path.exists():
        raise ValueError(f"Policy file not found: {doc_id}")
    content = policy.path.read_text(encoding="utf-8")
    return {"doc_id": doc_id, "title": policy.title, "content": content}

In [ ]:
# 문제 7-2. Prompt 템플릿

def incident_report_prompt(issue: str, affected_system: str) -> str:
    return f"""당신은 Acme Corp의 IT 운영 전문가입니다.
다음 인시던트에 대해 아래 형식으로 보고서를 한국어로 작성하세요.

이슈: {issue}
영향 시스템: {affected_system}

보고서 형식:
1. 요약: 이슈를 한 문장으로 요약
2. 영향 범위: 어떤 시스템과 사용자에게 영향을 미치는지
3. 재현 절차: 문제를 재현하는 단계별 절차
4. 권장 조치: 해결 또는 대응 방안
5. 우선순위: 긴급도 판단 (Critical/High/Medium/Low)"""

In [ ]:
# 검증 7

# Resource 테스트
tmp = Path(tempfile.mkdtemp())
(tmp / "vpn-setup.md").write_text(POLICY_DOC_1, encoding="utf-8")
(tmp / "security-policy.md").write_text(POLICY_DOC_2, encoding="utf-8")

policies = [
    PolicyDoc("vpn-setup", "VPN 설정 가이드", tmp / "vpn-setup.md", ["vpn", "네트워크"]),
    PolicyDoc("security-policy", "보안 정책", tmp / "security-policy.md", ["보안", "인증"]),
]

index = build_policy_index(policies)
print(f"인덱스: {len(index)}개")
print(json.dumps(index, ensure_ascii=False, indent=2))

detail = get_policy_content(policies, "vpn-setup", tmp)
print(f"\n상세: {detail['doc_id']} - {detail['title']} ({len(detail['content'])}자)")

try:
    get_policy_content(policies, "../../etc/passwd", tmp)
except ValueError:
    print("Path Traversal 차단됨")

# Prompt 테스트
prompt = incident_report_prompt("2층 프린터 작동 중단", "HP LaserJet Pro M404dn")
for item in ["요약", "영향 범위", "재현 절차", "권장 조치", "우선순위"]:
    print(f"  {item}: {'✓' if item in prompt else '✗'}")
print(f"  이슈 포함: {'✓' if '프린터' in prompt else '✗'}")
print(f"  시스템 포함: {'✓' if 'LaserJet' in prompt else '✗'}")

# 기대 출력:
# 인덱스: 2개
# [{"doc_id": "vpn-setup", ...}, {"doc_id": "security-policy", ...}]
# 상세: vpn-setup - VPN 설정 가이드
# Path Traversal 차단됨
# 요약: ✓ / 영향 범위: ✓ / ... 모두 ✓

---

## Part 2 (킬러 30점) — 미풀이

- Part 2는 MCP 서버 구동·Inspector 캡처·curl·pytest 증빙이 필요한 실전 과제로,
  노트북 단독으로 풀이할 수 없어 본 파일에서는 생략합니다.
- 별도 , ,  폴더 + 캡처 제출이 필요합니다.
